# Notebook 2: Data Preprocessing & Dimensionality Reduction

In this notebook I clean the dataset, drop unnecessary columns, and prepare the data for clustering.
I use insights from the EDA to decide which columns to keep or remove.

In [ ]:
# Importing libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler

# Load the raw training data (one level up from notebooks/ folder)
df = pd.read_csv('../train.csv')
print("Original shape:", df.shape)
df.head()

## 1. Drop Unnecessary Columns

Based on EDA, I will drop the following:
- `Unnamed: 0` — just a row index, not useful
- `lyrics` — raw text, not needed for numeric clustering (as instructed)
- `artist_name`, `track_name`, `release_date`, `genre`, `topic` — these are identifiers/labels, not numeric features for clustering

I'll save the identifiers in a separate dataframe so I can still reference them later.

In [30]:
# Save identifiers for reference later
id_cols = ['artist_name', 'track_name', 'release_date', 'genre', 'topic']
df_ids = df[id_cols].copy()

# Drop non-numeric and non-useful columns
cols_to_drop = ['Unnamed: 0', 'lyrics', 'artist_name', 'track_name',
                'release_date', 'genre', 'topic']
df_clean = df.drop(columns=cols_to_drop)

print("Shape after dropping columns:", df_clean.shape)
df_clean.head()

Shape after dropping columns: (28362, 17)


,len,dating,violence,world/life,night/time,shake the audience,family/gospel,romantic,communication,obscene,music,movement/places,light/visual perceptions,family/spiritual,sadness,feelings,age
0,95,0.000598,0.063746,0.000598,0.000598,0.000598,0.048857,0.017104,0.263751,0.000598,0.039288,0.000598,0.000598,0.000598,0.380299,0.117175,1.0
1,51,0.035537,0.096777,0.443435,0.001284,0.001284,0.027007,0.001284,0.001284,0.001284,0.118034,0.001284,0.212681,0.051124,0.001284,0.001284,1.0
2,24,0.002770,0.002770,0.002770,0.002770,0.002770,0.002770,0.158564,0.250668,0.002770,0.323794,0.002770,0.002770,0.002770,0.002770,0.225422,1.0
3,54,0.048249,0.001548,0.001548,0.001548,0.021500,0.001548,0.411536,0.001548,0.001548,0.001548,0.129250,0.001548,0.001548,0.225889,0.001548,1.0
4,48,0.001350,0.001350,0.417772,0.001350,0.001350,0.001350,0.463430,0.001350,0.001350,0.001350,0.001350,0.001350,0.029755,0.068800,0.001350,1.0


## 2. Check for Missing Values and Outliers

In [31]:
# Confirm no missing values remain
print("Missing values:")
print(df_clean.isnull().sum())

# Check for extreme outliers using IQR
print()
print("Checking numeric ranges:")
print(df_clean.describe())

Missing values:
len                         0
dating                      0
violence                    0
world/life                  0
night/time                  0
shake the audience          0
family/gospel               0
romantic                    0
communication               0
obscene                     0
music                       0
movement/places             0
light/visual perceptions    0
family/spiritual            0
sadness                     0
feelings                    0
age                         0
dtype: int64

Checking numeric ranges:
                len        dating      violence    world/life    night/time  \
count  28362.000000  28362.000000  28362.000000  28362.000000  28362.000000   
mean      73.030534      0.021110      0.118371      0.120984      0.057356   
std       41.831605      0.052366      0.178658      0.172216      0.111892   
min        1.000000      0.000291      0.000284      0.000291      0.000289   
25%       42.000000      0.000923      0

In [ ]:
# Check the top word counts to understand the range of outliers
print("Top 10 word counts:")
print(df_clean['len'].nlargest(10))

# Remove songs above the 99th percentile — removes extreme outliers without losing too much data
len_threshold = df_clean['len'].quantile(0.99)
print(f"\n99th percentile for word count: {len_threshold}")

# Keep ids in sync when filtering rows
df_clean = df_clean[df_clean['len'] <= len_threshold]
df_ids = df_ids[df['len'] <= len_threshold]

print("Shape after removing outliers:", df_clean.shape)

## 3. Check Correlation ,Drop Highly Correlated Columns

From the EDA, I noticed some features are highly correlated.
If two columns are correlated above 0.85, I'll drop one to reduce redundancy.

In [ ]:
# Use absolute values so direction of correlation doesn't matter
corr_matrix = df_clean.corr().abs()

# Only look at the upper triangle to avoid counting each pair twice
upper_tri = corr_matrix.where(
    np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
)

# Drop any column where at least one correlation exceeds 0.85
to_drop = [col for col in upper_tri.columns if any(upper_tri[col] > 0.85)]
print("Highly correlated columns to drop:", to_drop)

df_clean = df_clean.drop(columns=to_drop)
print("Shape after dropping correlated columns:", df_clean.shape)

## 4. Scale the Features

KMeans clustering is sensitive to the scale of features.
I'll use StandardScaler to normalize everything so no single feature dominates.

In [ ]:
# KMeans uses Euclidean distance so all features must be on the same scale
# StandardScaler transforms each column to mean=0 and std=1
scaler = StandardScaler()
df_scaled_array = scaler.fit_transform(df_clean)

# Wrap back in a DataFrame to preserve column names
df_scaled = pd.DataFrame(df_scaled_array, columns=df_clean.columns)

print("Scaled data shape:", df_scaled.shape)
df_scaled.head()

## 5. Save Cleaned Data

I'll save two files:
- `train_cleaned.csv` — the scaled numeric features used for modeling
- `train_ids.csv` — the song identifiers to reference cluster results

In [ ]:
import pickle

# Save the cleaned and scaled numeric features
df_scaled.to_csv('../data/train_cleaned.csv', index=False)

# Reset index before saving ids to match
df_ids = df_ids.reset_index(drop=True)
df_ids.to_csv('../data/train_ids.csv', index=False)

# Save the scaler so notebook 4 can apply identical scaling to test data
with open('../data/scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

print("Saved train_cleaned.csv, train_ids.csv, and scaler.pkl")
print("Final feature set used for modeling:")
print(df_scaled.columns.tolist())